# Brazil 2026 Presidential Election — Round Comparison Map

**Metric:** `(Lula 2° turno − Lula 1° turno) − (Flávio 2° turno − Flávio 1° turno)`

Positive (red): Lula gained more votes between rounds  
Negative (blue): Flávio gained more votes between rounds

In [ ]:
import os, io, zipfile, requests
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## 1. Download TSE Vote Data

In [ ]:
def download_if_missing(url, dest):
    if dest.exists():
        print(f"Using cached: {dest}")
        return
    print(f"Downloading {url} ...")
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)
    print(f"Saved to {dest}")

# TSE 2026 - votes by candidate, municipality and zone
TSE_URL = "https://cdn.tse.jus.br/estatistica/sead/odsele/votacao_candidato_munzona/votacao_candidato_munzona_2026.zip"
TSE_ZIP = DATA_DIR / "votacao_candidato_munzona_2026.zip"
download_if_missing(TSE_URL, TSE_ZIP)

In [ ]:
# Read CSV from zip - TSE uses latin-1 encoding and semicolon separator
with zipfile.ZipFile(TSE_ZIP) as z:
    csv_names = [n for n in z.namelist() if n.endswith('.csv')]
    print("Files in zip:", csv_names)
    # Read all CSV files (there may be one per state or one total)
    dfs = []
    for name in csv_names:
        with z.open(name) as f:
            df = pd.read_csv(f, encoding='latin-1', sep=';', dtype=str, low_memory=False)
            dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"Shape: {df_all.shape}")
print("Columns:", df_all.columns.tolist())
print(df_all.head(3))

In [ ]:
# Find relevant columns
print("Unique NR_TURNO:", df_all['NR_TURNO'].unique() if 'NR_TURNO' in df_all.columns else "column not found")

# Find candidate name column
name_col = 'NM_URNA_CANDIDATO' if 'NM_URNA_CANDIDATO' in df_all.columns else 'NM_CANDIDATO'
print(f"\nUsing name column: {name_col}")

# Filter to presidential race (CD_CARGO == '1' for President)
cargo_col = 'CD_CARGO' if 'CD_CARGO' in df_all.columns else 'DS_CARGO'
print(f"\nUnique cargos: {df_all[cargo_col].unique()[:20]}")

In [ ]:
# Presidential candidates: CD_CARGO = 1
df_pres = df_all[df_all['CD_CARGO'].astype(str).str.strip() == '1'].copy()
print(f"Presidential rows: {len(df_pres)}")

# Find Lula and Flavio
cands = df_pres[[name_col, 'NR_CANDIDATO', 'NR_TURNO']].drop_duplicates()
print("\nAll presidential candidates:")
print(cands.sort_values(['NR_TURNO', name_col]).to_string())

In [ ]:
# Identify candidate numbers from names
df_pres[name_col] = df_pres[name_col].str.upper().str.strip()
lula_mask = df_pres[name_col].str.contains('LULA', na=False)
flavio_mask = df_pres[name_col].str.contains('FLÁVIO|FLAVIO|BOLSONARO', na=False)

lula_name = df_pres.loc[lula_mask, name_col].iloc[0] if lula_mask.any() else None
flavio_name = df_pres.loc[flavio_mask, name_col].iloc[0] if flavio_mask.any() else None
print(f"Lula candidate: {lula_name}")
print(f"Flávio candidate: {flavio_name}")

# Vote column
vote_col = 'QT_VOTOS_NOMINAIS' if 'QT_VOTOS_NOMINAIS' in df_pres.columns else 'QT_VOTOS_NOMINAIS_VALIDOS'
mun_col = 'CD_MUNICIPIO' if 'CD_MUNICIPIO' in df_pres.columns else 'SG_MUNICIPIO'
print(f"\nVote column: {vote_col}")
print(f"Municipality column: {mun_col}")

# Convert vote column to numeric
df_pres[vote_col] = pd.to_numeric(df_pres[vote_col].str.replace(',', '.'), errors='coerce').fillna(0)
df_pres['NR_TURNO'] = df_pres['NR_TURNO'].astype(str).str.strip()

# Aggregate by municipality and round
def get_candidate_votes(name_pattern):
    mask = df_pres[name_col].str.contains(name_pattern, na=False)
    return (df_pres[mask]
            .groupby([mun_col, 'NR_TURNO'])[vote_col]
            .sum()
            .unstack('NR_TURNO')
            .rename(columns={'1': 'r1', '2': 'r2'}))

lula_votes = get_candidate_votes('LULA')
flavio_votes = get_candidate_votes('FLÁVIO|FLAVIO|BOLSONARO')

print(f"\nLula municipalities: {len(lula_votes)}")
print(f"Flávio municipalities: {len(flavio_votes)}")

# Merge and compute metric
df_metric = lula_votes.add_prefix('lula_').join(flavio_votes.add_prefix('flavio_'), how='outer')
df_metric.columns = [c.replace('lula_r', 'lula').replace('flavio_r', 'flavio') for c in df_metric.columns]
df_metric = df_metric.rename(columns={'lula1': 'lula_1', 'lula2': 'lula_2', 'flavio1': 'flavio_1', 'flavio2': 'flavio_2'})
df_metric = df_metric.fillna(0)

df_metric['metric'] = (df_metric['lula_2'] - df_metric['lula_1']) - (df_metric['flavio_2'] - df_metric['flavio_1'])
df_metric['total_votes'] = df_metric[['lula_1', 'lula_2', 'flavio_1', 'flavio_2']].max(axis=1)
df_metric['metric_pct'] = np.where(df_metric['total_votes'] > 0,
                                    df_metric['metric'] / df_metric['total_votes'] * 100, 0)

print(df_metric.describe())
df_metric = df_metric.reset_index()
print(df_metric.head())

## 2. Download Municipality Geometries

In [ ]:
GEO_URL = "https://servicodados.ibge.gov.br/api/v3/malhas/paises/BR?formato=application/vnd.geo+json&qualidade=minima&divisao=municipio"
GEO_FILE = DATA_DIR / "municipios.geojson"
download_if_missing(GEO_URL, GEO_FILE)

gdf = gpd.read_file(GEO_FILE)
print(f"GeoDataFrame shape: {gdf.shape}")
print("Columns:", gdf.columns.tolist())
print(gdf.head(3))

In [ ]:
STATE_URL = "https://servicodados.ibge.gov.br/api/v3/malhas/paises/BR?formato=application/vnd.geo+json&qualidade=minima&divisao=estado"
STATE_FILE = DATA_DIR / "estados.geojson"
download_if_missing(STATE_URL, STATE_FILE)
gdf_states = gpd.read_file(STATE_FILE)
print(gdf_states.head())

In [ ]:
# IBGE municipality codes are 7 digits; TSE uses 5 digits
# TSE CD_MUNICIPIO maps to the first 6 chars of IBGE code... 
# Actually TSE uses its own 5-digit code, not IBGE code.
# IBGE codarea is 7 digits. TSE CD_MUNICIPIO is 5 digits.
# The relationship: pad TSE to 6 digits or use name matching.
# Try: IBGE 7-digit code, first 6 digits = state+municipality IBGE code (without digit verificador)
# TSE CD_MUNICIPIO is a completely different code system.
# Best approach: use municipality name matching or a crosswalk.

# Check what the geometry id column looks like
id_col = gdf.columns[0]  # usually 'codarea' or 'id'
print(f"Geometry id column: {id_col}")
print("Sample ids:", gdf[id_col].head(10).tolist())
print("Sample TSE codes:", df_metric[mun_col].head(10).tolist())

In [ ]:
# The IBGE GeoJSON uses 7-digit IBGE municipality codes in 'codarea'
# TSE uses its own 5-digit CD_MUNICIPIO codes
# We need a crosswalk. The TSE open data files include a 'municipios.csv' or similar,
# or we can download from TSE: https://cdn.tse.jus.br/estatistica/sead/odsele/auxiliary_data/

# Alternative: use the NM_MUNICIPIO and SG_UF from TSE data to match by name+state
# Let's get municipality names from the TSE data
mun_info = df_pres[[mun_col, 'NM_MUNICIPIO', 'SG_UF']].drop_duplicates() if 'NM_MUNICIPIO' in df_pres.columns else df_pres[[mun_col, 'SG_UF']].drop_duplicates()
print(mun_info.head())

# Try direct numeric join (TSE 5-digit vs IBGE 7-digit)
# IBGE codarea: first 2 digits = state code, next 5 = municipality = that's the relationship!
gdf['tse_code'] = gdf['codarea'].astype(str).str[:-1].str.lstrip('0')  # remove last digit (verificador) and leading zeros -- actually try str[1:6] 
# TSE codes: numeric, no leading zeros typically
df_metric[mun_col + '_str'] = df_metric[mun_col].astype(str).str.zfill(5)
gdf['codarea_6'] = gdf['codarea'].astype(str).str[:6]

# Check overlap
print("Sample gdf codarea:", gdf['codarea'].head().tolist())
print("Sample tse codes:", df_metric[mun_col].head().tolist())

In [ ]:
# IBGE 7-digit code: first 6 digits match TSE's "extended" code
# TSE CD_MUNICIPIO is 5 chars, IBGE is 7 chars
# The mapping: IBGE_7digit[0:6] typically = state_2digit + municipality_4digit
# while TSE = different encoding
# Let's try: merge on first 5 digits of IBGE dropping the check digit
gdf['ibge_5'] = gdf['codarea'].astype(str).str[:5]
df_metric['tse_5'] = df_metric[mun_col].astype(str).str.zfill(5)

merged = gdf.merge(df_metric, left_on='ibge_5', right_on='tse_5', how='left')
print(f"Merged: {len(merged)} rows, {merged['metric'].notna().sum()} with metric data")

# If poor match, try 6-digit comparison
if merged['metric'].notna().sum() < 100:
    gdf['ibge_6'] = gdf['codarea'].astype(str).str[:6]
    df_metric['tse_6'] = df_metric[mun_col].astype(str).str.zfill(6)
    merged = gdf.merge(df_metric, left_on='ibge_6', right_on='tse_6', how='left')
    print(f"Retry with 6 digits: {merged['metric'].notna().sum()} matches")

print(merged[['codarea', 'metric', 'metric_pct']].head(10))

## 3. Create the Map

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(16, 18), facecolor='#1a1a2e')
ax.set_facecolor('#1a1a2e')

# Determine color scale
max_val = merged['metric'].abs().quantile(0.98)
norm = TwoSlopeNorm(vmin=-max_val, vcenter=0, vmax=max_val)

# Custom diverging colormap: blue (Flávio gains) → white → red (Lula gains)
cmap = plt.cm.RdBu_r  # Red-Blue reversed: red=positive=Lula, blue=negative=Flávio

# Plot municipalities
merged.plot(
    column='metric',
    ax=ax,
    cmap=cmap,
    norm=norm,
    linewidth=0.05,
    edgecolor='#1a1a2e',
    missing_kwds={'color': '#2a2a4e', 'label': 'Sem dados'}
)

# Overlay state boundaries
gdf_states.boundary.plot(ax=ax, color='white', linewidth=0.5, alpha=0.6)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.4, aspect=20, pad=0.02)
cbar.ax.tick_params(colors='white', labelsize=9)
cbar.set_label('Votos', color='white', fontsize=10)
cbar.outline.set_edgecolor('white')

# Format colorbar ticks
tick_vals = [-max_val, -max_val/2, 0, max_val/2, max_val]
cbar.set_ticks(tick_vals)
cbar.set_ticklabels([f'{v:+,.0f}' for v in tick_vals])

# Annotations
ax.set_title(
    '(Lula 2° − Lula 1°) − (Flávio 2° − Flávio 1°)',
    color='white', fontsize=18, fontweight='bold', pad=15
)
fig.text(0.5, 0.935, 'Variação Relativa de Votos por Município — Eleições 2026',
         ha='center', color='#cccccc', fontsize=13)
fig.text(0.5, 0.915, 
         'Vermelho: Lula ganhou mais votos do 1° para o 2° turno  |  Azul: Flávio ganhou mais',
         ha='center', color='#aaaaaa', fontsize=10, style='italic')

ax.axis('off')
plt.tight_layout(rect=[0, 0.02, 1, 0.92])

fig.text(0.5, 0.01, 'Fonte: TSE — Tribunal Superior Eleitoral', 
         ha='center', color='#666666', fontsize=8)

plt.savefig('election_map.png', dpi=200, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("Saved election_map.png")